In [1]:
# ============================================================
# NOTEBOOK 8 — TRANSACTION-LEVEL MERGE
# Bring all segmentation columns from customer-level back onto
# the transaction-level master file. One left join on user_id.
# ============================================================

In [2]:
import pandas as pd
import os

In [3]:
# ------------------------------------------------------------
# PATH SETUP & LOAD CHECKPOINTS
# ------------------------------------------------------------
path = r'/Users/elia/Desktop/[02] Data Projects/[2] Working Folder/[3] InstaCart Store'

In [4]:
# Load the master transaction-level file
df_master = pd.read_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'instacart_master.pkl')
)
print(f"Master loaded: {df_master.shape}")

Master loaded: (32398592, 21)


In [5]:
# Load the customer-level segmentation table
df_customers_agg = pd.read_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'customers_agg_step7.pkl')
)
print(f"Customer aggregation loaded: {df_customers_agg.shape}")
print()

Customer aggregation loaded: (206209, 27)



In [6]:
# ------------------------------------------------------------
# 8A — IDENTIFY COLUMNS TO MERGE
# ------------------------------------------------------------
# We don't need to merge demographic columns that are already
# in df_master (gender, state, age, n_dependants, fam_status,
# income, date_joined). We bring across only the new derived
# columns that customer aggregation produced.

In [7]:
# Columns NOT to merge (already in master OR not needed at line level)
already_in_master = {'gender', 'state', 'age', 'date_joined',
                     'n_dependants', 'fam_status', 'income'}

In [8]:
# user_id is the merge key
merge_key = 'user_id'

In [9]:
# Everything else gets merged in
cols_to_merge = [merge_key] + [
    col for col in df_customers_agg.columns
    if col not in already_in_master and col != merge_key
]

print(f"Columns to merge from customer-level → transaction-level:")
print(f"  Total: {len(cols_to_merge) - 1} new columns (plus merge key)")
print(f"  {[c for c in cols_to_merge if c != merge_key]}")
print()

df_to_merge = df_customers_agg[cols_to_merge].copy()

Columns to merge from customer-level → transaction-level:
  Total: 19 new columns (plus merge key)
  ['unique_orders', 'unique_products', 'avg_product_price', 'total_reordered_items', 'total_line_items', 'reorder_rate', 'orders_pct_rank', 'reorder_pct_rank', 'products_pct_rank', 'price_pct_rank', 'behaviour_score', 'priority_level', 'behaviour_segment', 'region', 'age_group', 'income_group', 'household_group', 'demographic_profile', 'strategic_target_segment']



In [10]:
# ------------------------------------------------------------
# 8B — VERIFY MERGE PRECONDITIONS
# ------------------------------------------------------------

In [11]:
# user_id must be unique in the customer-level file
assert df_to_merge[merge_key].duplicated().sum() == 0, \
    "user_id is not unique in df_customers_agg — merge would inflate rows"

In [12]:
# Every user_id in master must exist in customer-level
master_users   = set(df_master['user_id'].unique())
customer_users = set(df_to_merge['user_id'].unique())
missing_users  = master_users - customer_users
assert len(missing_users) == 0, \
    f"⚠️ {len(missing_users)} user_ids in master have no segmentation record"

print(f"✅ Preconditions verified")
print(f"   Unique users in master: {len(master_users):,}")
print(f"   Unique users in segmentation: {len(customer_users):,}")
print()

✅ Preconditions verified
   Unique users in master: 206,209
   Unique users in segmentation: 206,209



In [14]:
# ------------------------------------------------------------
# 8C — MERGE
# ------------------------------------------------------------
# Left join on user_id. Master row count must be preserved exactly.

rows_before = df_master.shape[0]
cols_before = df_master.shape[1]

df_master_segmented = df_master.merge(
    df_to_merge,
    on=merge_key,
    how='left'
)

rows_after = df_master_segmented.shape[0]
cols_after = df_master_segmented.shape[1]

In [15]:
# Row count assertion — the brief's hard requirement
assert rows_after == rows_before, \
    f"Row count changed: {rows_before:,} → {rows_after:,}"

# Column count check
assert cols_after == cols_before + (len(cols_to_merge) - 1), \
    "Column count after merge is unexpected"

# No nulls in any segmentation column (every transaction must have a segment)
new_cols = [c for c in cols_to_merge if c != merge_key]
for col in new_cols:
    null_count = df_master_segmented[col].isnull().sum()
    assert null_count == 0, f"⚠️ {col} has {null_count} nulls after merge"

print(f"✅ Merge complete")
print(f"   Rows preserved: {rows_after:,} (unchanged)")
print(f"   Columns: {cols_before} → {cols_after}")
print(f"   No nulls in any segmentation column")
print()

✅ Merge complete
   Rows preserved: 32,398,592 (unchanged)
   Columns: 21 → 40
   No nulls in any segmentation column



In [16]:
# ------------------------------------------------------------
# 8D — VALIDATION
# ------------------------------------------------------------
# Cross-check: pick 3 random users and verify their segmentation
# values are identical at the transaction level vs customer level.

In [17]:
sample_users = df_customers_agg['user_id'].sample(3, random_state=42).tolist()
print(f"Spot-check {len(sample_users)} random users — values must match exactly:")
for uid in sample_users:
    cust_row  = df_customers_agg[df_customers_agg['user_id'] == uid].iloc[0]
    txn_rows  = df_master_segmented[df_master_segmented['user_id'] == uid]
    
    # Check 4 key columns
    assert (txn_rows['priority_level']           == cust_row['priority_level']).all()
    assert (txn_rows['behaviour_segment']        == cust_row['behaviour_segment']).all()
    assert (txn_rows['strategic_target_segment'] == cust_row['strategic_target_segment']).all()
    assert (txn_rows['region']                   == cust_row['region']).all()
    
    print(f"   user_id {uid}: {len(txn_rows)} txns, "
          f"strategic={cust_row['strategic_target_segment']}, "
          f"priority={cust_row['priority_level']}")
print()

Spot-check 3 random users — values must match exactly:
   user_id 189033: 76 txns, strategic=Premium Higher Price Buyers, priority=Maintain
   user_id 113007: 6 txns, strategic=Reactivate Low-Engagement Families, priority=Low Priority
   user_id 40369: 204 txns, strategic=Grow Active Family Shoppers, priority=Growth Priority



In [18]:
# ------------------------------------------------------------
# 8E — SAVE
# ------------------------------------------------------------

In [19]:
output_path = os.path.join(path, '[2] Data', '[2] Prepared Data',
                           'instacart_master_segmented.pkl')

df_master_segmented.to_pickle(output_path)

print(f"✅ NOTEBOOK 8 COMPLETE — saved: instacart_master_segmented.pkl")
print(f"   Final shape: {df_master_segmented.shape}")
print(f"   Memory size: ~{df_master_segmented.memory_usage(deep=True).sum() / 1e9:.1f} GB")
print()
print(f"All columns:")
for i, col in enumerate(df_master_segmented.columns, 1):
    print(f"   {i:>2}. {col}")

✅ NOTEBOOK 8 COMPLETE — saved: instacart_master_segmented.pkl
   Final shape: (32398592, 40)
   Memory size: ~21.6 GB

All columns:
    1. order_id
    2. user_id
    3. order_number
    4. order_day_of_week
    5. order_hour_of_day
    6. days_since_prior_order
    7. product_id
    8. add_to_cart_order
    9. reordered
   10. product_name
   11. aisle_id
   12. department_id
   13. prices
   14. department_name
   15. gender
   16. state
   17. age
   18. date_joined
   19. n_dependants
   20. fam_status
   21. income
   22. unique_orders
   23. unique_products
   24. avg_product_price
   25. total_reordered_items
   26. total_line_items
   27. reorder_rate
   28. orders_pct_rank
   29. reorder_pct_rank
   30. products_pct_rank
   31. price_pct_rank
   32. behaviour_score
   33. priority_level
   34. behaviour_segment
   35. region
   36. age_group
   37. income_group
   38. household_group
   39. demographic_profile
   40. strategic_target_segment
